<a href="https://colab.research.google.com/github/win-eva/als-sex-stratified-target-discovery/blob/main/04_ml_script.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests, time, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.sparse import csr_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import BaggingClassifier, RandomForestRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression, ElasticNetCV
from xgboost import XGBRegressor
from google.colab import drive

warnings.filterwarnings("ignore")
drive.mount("/content/drive")
BASE = Path("/content/drive/MyDrive/ALS Data")

Mounted at /content/drive


In [ ]:
FEATURE_COLS = ["reversal_score", "chembl_n_compounds", "binary_pchembl",
                "sex_consistency", "n_independent_sources", "coexpr_score", "cns_expressed"]

OT_GRAPHQL = "https://api.platform.opentargets.org/api/v4/graphql"
STRING_API = "https://string-db.org/api"
OT_SEED_CUTOFF = 0.3       # OT ALS-association score above which a gene becomes a strong PU seed
PROPAGATION_ALPHA = 0.5

RF_TUNED = dict(n_estimators=500, max_depth=4, min_samples_leaf=10,
                 max_features="sqrt", random_state=42, n_jobs=-1)
XGB_TUNED = dict(n_estimators=100, max_depth=2, learning_rate=0.03,
                  subsample=0.8, colsample_bytree=0.8,
                  random_state=42, n_jobs=-1, verbosity=0)

# well-characterised cancer targets (flagged rather than removed, since CLUE's
# oncology bias means they can dominate a ranking on reversal signal alone)
CANCER_GENES = {"TP53", "KIT", "PDGFRA", "PDGFRB", "EGFR", "AKT1", "ABL1", "SRC",
                 "JAK1", "JAK2", "JAK3", "CDK1", "CDK2", "AURKA", "AURKB", "IGF1R",
                 "RAF1", "MCL1", "MELK", "FLT3", "CSF1R", "MAP2K1"}

# non-gene junk that occasionally slips through as a "gene symbol"
ARTIFACTS = {"HSP90AB2P", "METG", "NCE103", "1272966", "U38", "MEX-5", "POS-1",
             "GROS", "GROL", "GPM3", "BLAACC-4", "BLAPER-2", "PPO4", "PPO1",
             "PPO3", "RPMD", "RPMF", "RPLN"}

## Load targets and cached annotations

`chembl_n_compounds_cache.csv` and `hpa_ot_cns_cache.csv` are pre-fetched (ChEMBL
compound counts and Human Protein Atlas / Open Targets CNS-expression flags per gene)
rather than queried fresh each run.

In [ ]:
chembl_counts = pd.read_csv(BASE / "chembl_n_compounds_cache.csv", index_col=0, comment="#").squeeze()
cns_flags = pd.read_csv(BASE / "hpa_ot_cns_cache.csv", index_col=0).squeeze()

def load_targets(path, sex):
    df = pd.read_csv(path, index_col="rank")
    df["sex"] = sex
    df["target_gene"] = df["target_gene"].str.strip().str.upper()
    return (
        df.sort_values("reversal_score", ascending=False)
        .drop_duplicates(subset="target_gene", keep="first")
        .reset_index(drop=True)
    )

def count_independent_sources(s):
    """CLUE and ChEMBL are the two annotation sources; ChEMBL's various lookup
    routes (name, InChIKey, SMILES) all collapse to one "chembl" source."""
    if pd.isna(s):
        return 1
    sources = set(x.strip() for x in str(s).split(";"))
    chembl_variants = {"chembl", "chembl_inchikey", "chembl_smiles", "opentargets"}
    if sources & chembl_variants:
        sources = (sources - chembl_variants) | {"chembl"}
    return max(1, len(sources))

female_all = load_targets(BASE / "als_targets_female.csv", "female")
male_all = load_targets(BASE / "als_targets_male.csv", "male")
print(f"Female: {len(female_all)} | Male: {len(male_all)}")

## Co-expression score

STRING network proximity to known ALS genes is the standard proximity feature in
target-prioritisation pipelines, but STRING is also used below to define the PU
learning labels. Using it as a feature too would let the model partly reward genes
for the same property that got them labelled positive in the first place. Co-expression
computed directly from the patient RNA-seq data gives a proximity measure that's
independent of that labelling step.

In [ ]:
counts = pd.read_csv(BASE / "ALS_counts_pre_normalisation_all_donors_collapsed.csv")
meta_cols = ["Geneid", "symbol", "biotype", "description"]
donor_cols = [c for c in counts.columns if c not in meta_cols]

gene_symbols = counts["symbol"].values
expr = counts[donor_cols].values.astype(float)
lib_sizes = expr.sum(axis=0)
lib_sizes[lib_sizes == 0] = 1
log_cpm = np.log2(expr / lib_sizes[np.newaxis, :] * 1e6 + 1)
symbol_to_idx = {s: i for i, s in enumerate(gene_symbols)}

ALS_SEED_GENES = ["SOD1", "TARDBP", "FUS", "TBK1", "NEK1", "SQSTM1",
                   "UBQLN2", "VCP", "OPTN", "ANG", "ATXN2", "KIF5A"]
seed_idx = [symbol_to_idx[g] for g in ALS_SEED_GENES if g in symbol_to_idx]
seed_matrix = log_cpm[seed_idx, :]

novel_genes = list(set(
    female_all[female_all["pathway_tier"] == "novel"]["target_gene"].tolist() +
    male_all[male_all["pathway_tier"] == "novel"]["target_gene"].tolist()
))

def max_seed_correlation(gene_expr, seed_matrix):
    """Max |Spearman r| against the 12 ALS seed genes. Genes with zero variance
    (undetected in this cohort) correlate with nothing by definition."""
    correlations = []
    for seed_row in seed_matrix:
        if np.std(gene_expr) == 0 or np.std(seed_row) == 0:
            correlations.append(0)
            continue
        r = pd.Series(gene_expr).corr(pd.Series(seed_row), method="spearman")
        correlations.append(abs(r))
    return max(correlations)

coexpr_scores = {
    g: max_seed_correlation(log_cpm[symbol_to_idx[g], :], seed_matrix) if g in symbol_to_idx else 0.0
    for g in novel_genes
}
coexpr_series = pd.Series(coexpr_scores)
print(f"Co-expression scores computed for {len(coexpr_series)} genes")

## Open Targets ALS association score (regression label)

In [ ]:
all_rows, page = [], 0
while True:
    query = """{ disease(efoId: "MONDO_0004976") {
        associatedTargets(page: {index: %d, size: 500}) {
          count rows { target { approvedSymbol } score } } } }""" % page
    for _ in range(3):
        try:
            r = requests.post(OT_GRAPHQL, json={"query": query}, timeout=60)
            break
        except requests.RequestException:
            time.sleep(5)
    data = r.json()["data"]["disease"]["associatedTargets"]
    all_rows.extend({"target_gene": row["target"]["approvedSymbol"].upper(), "ot_overall": row["score"]}
                     for row in data["rows"])
    if len(all_rows) >= data["count"]:
        break
    page += 1
    time.sleep(0.2)

ot_scores = (
    pd.DataFrame(all_rows)
    .sort_values("ot_overall", ascending=False)
    .drop_duplicates(subset="target_gene", keep="first")
    .set_index("target_gene")
)
print(f"Open Targets scores retrieved for {len(ot_scores)} genes")

## STRING network (for PU labels only)

Genes with an Open Targets ALS score above the seed cutoff become "strong" positive
labels. STRING first-degree neighbours of the 12 specific ALS genes become "weak" positive
labels when they also have some non-zero OT signal. STRING itself never enters
`FEATURE_COLS`.

In [ ]:
seed_genes = ot_scores[ot_scores["ot_overall"] >= OT_SEED_CUTOFF].index.tolist()
all_genes = list(set(female_all["target_gene"].tolist() + male_all["target_gene"].tolist() + seed_genes))

all_edges, prior = [], 0.041  # STRING's background prior for the combined score correction
for i in range(0, len(all_genes), 100):
    batch = all_genes[i:i + 100]
    for _ in range(3):
        try:
            r = requests.post(
                f"{STRING_API}/json/network",
                data={"identifiers": "\r".join(batch), "species": 9606,
                      "required_score": 150, "caller_identity": "als_ucl_masters"},
                timeout=120,
            )
            if r.status_code == 200:
                for interaction in r.json():
                    a, e = float(interaction.get("ascore", 0)), float(interaction.get("escore", 0))
                    combined = 1 - (1 - a) * (1 - e)
                    corrected = (combined - prior) / (1 - prior) if combined > prior else 0
                    if corrected >= 0.05:
                        all_edges.append({
                            "gene_a": interaction.get("preferredName_A", "").upper(),
                            "gene_b": interaction.get("preferredName_B", "").upper(),
                            "score": corrected,
                        })
                break
        except requests.RequestException:
            time.sleep(15)
    time.sleep(2)

edges = pd.DataFrame(all_edges)
edges = edges[edges["gene_a"] != edges["gene_b"]].drop_duplicates(subset=["gene_a", "gene_b"])
print(f"STRING edges: {len(edges)}")

top_als_set = set(ALS_SEED_GENES)
first_degree = set()
for _, e in edges.iterrows():
    if e["gene_a"] in top_als_set:
        first_degree.add(e["gene_b"])
    if e["gene_b"] in top_als_set:
        first_degree.add(e["gene_a"])
first_degree -= top_als_set
print(f"First-degree neighbours of the 12 seed genes: {len(first_degree)}")

## Feature matrix and PU labels

In [ ]:
def build_features(targets, female_all, male_all, ot_scores, coexpr_series, sex,
                    chembl_counts, cns_flags, string_first_degree=None):
    string_first_degree = string_first_degree or set()
    df = targets[targets["pathway_tier"] == "novel"].copy()

    other = None
    if sex == "female":
        other = male_all.set_index("target_gene")["reversal_score"]
    elif sex == "male":
        other = female_all.set_index("target_gene")["reversal_score"]

    if other is not None:
        df["other_reversal"] = df["target_gene"].map(other).fillna(0)
        df["sex_consistency"] = df.apply(
            lambda r: min(r["reversal_score"], r["other_reversal"]) / max(r["reversal_score"], r["other_reversal"])
            if max(r["reversal_score"], r["other_reversal"]) > 0 else 0, axis=1
        )
        df.loc[df["other_reversal"] == 0, "sex_consistency"] = 0
    else:
        df["sex_consistency"] = 0

    df["ot_overall"] = df["target_gene"].map(ot_scores["ot_overall"]).fillna(0)
    df["mean_pchembl"] = pd.to_numeric(df["mean_pchembl"], errors="coerce").fillna(0)
    df["binary_pchembl"] = (df["mean_pchembl"] >= 6.0).astype(int)
    df["reversal_score"] = pd.to_numeric(df["reversal_score"], errors="coerce").fillna(0)
    df["n_independent_sources"] = df["sources"].apply(count_independent_sources) if "sources" in df.columns else 1
    df["chembl_n_compounds"] = df["target_gene"].map(chembl_counts).fillna(0)
    df["cns_expressed"] = df["target_gene"].map(cns_flags).fillna(0).astype(int)
    df["coexpr_score"] = df["target_gene"].map(coexpr_series).fillna(0)

    strong_pos = set(df[df["ot_overall"] >= OT_SEED_CUTOFF]["target_gene"])
    weak_pos = set(df[df["target_gene"].isin(string_first_degree) & (df["ot_overall"] > 0)]["target_gene"])
    df["pu_label"] = df["target_gene"].isin(strong_pos | weak_pos).astype(int)
    df["pu_strength"] = df["target_gene"].apply(
        lambda g: "strong" if g in strong_pos else "weak" if g in weak_pos else "unlabelled"
    )
    return df.reset_index(drop=True)

female_df = build_features(female_all, female_all, male_all, ot_scores, coexpr_series,
                            "female", chembl_counts, cns_flags, string_first_degree=first_degree)
male_df = build_features(male_all, female_all, male_all, ot_scores, coexpr_series,
                          "male", chembl_counts, cns_flags, string_first_degree=first_degree)

# combined analysis: only genes novel in both sex-stratified analyses
female_novel = set(female_all[female_all["pathway_tier"] == "novel"]["target_gene"])
male_novel = set(male_all[male_all["pathway_tier"] == "novel"]["target_gene"])
true_intersection = female_novel & male_novel
combined_targets = pd.concat([
    female_all[female_all["target_gene"].isin(true_intersection)],
    male_all[male_all["target_gene"].isin(true_intersection)],
], ignore_index=True).drop_duplicates(subset="target_gene")

combined_df = build_features(combined_targets, female_all, male_all, ot_scores, coexpr_series,
                              "combined", chembl_counts, cns_flags, string_first_degree=first_degree)

datasets = {"female": female_df, "male": male_df, "combined": combined_df}
for sex, df in datasets.items():
    print(f"{sex}: n={len(df)}, positives={df['pu_label'].sum()}, coexpr_mean={df['coexpr_score'].mean():.3f}")

## Held-out validation set

Three-way check: do models actually separate known ALS-relevant genes (positive
controls) from random novel targets and from genes with no plausible ALS
connection (negative controls)?

In [ ]:
h_strong = ["PDGFRA", "FGFR3", "PDGFRB"]
h_weak = ["CHRNA4", "NTRK2", "LRRK2", "FYN", "CAMK4"]
pos_genes = h_strong + h_weak

bottom100 = datasets["combined"].nsmallest(100, "reversal_score")
neg_genes = bottom100[
    (bottom100["cns_expressed"] == 0) & (~bottom100["target_gene"].isin(ARTIFACTS))
]["target_gene"].tolist()

remaining = datasets["combined"][~datasets["combined"]["target_gene"].isin(pos_genes + neg_genes)]
h_random = remaining.sample(n=63, random_state=13)["target_gene"].tolist()
holdout_genes = pos_genes + h_random + neg_genes

print(f"Positive controls ({len(pos_genes)}): {pos_genes}")
print(f"Negative controls ({len(neg_genes)}): {neg_genes}")
print(f"Random comparison genes: {len(h_random)}")

## Model comparison

In [ ]:
feat_df = datasets["combined"]
train_df = feat_df[~feat_df["target_gene"].isin(holdout_genes)].copy()
test_df = feat_df[feat_df["target_gene"].isin(holdout_genes)].copy()

X_train, X_test = train_df[FEATURE_COLS].fillna(0).values, test_df[FEATURE_COLS].fillna(0).values
y_pu, y_ot = train_df["pu_label"].values, train_df["ot_overall"].values

scaler = StandardScaler()
X_train_s, X_test_s = scaler.fit_transform(X_train), scaler.transform(X_test)

n_pos = y_pu.sum()
bootstrap_frac = min(1.0, (2 * n_pos) / len(train_df))

models = {
    "pu_bag": (BaggingClassifier(
        estimator=DecisionTreeClassifier(max_depth=2, min_samples_leaf=5, class_weight="balanced"),
        n_estimators=100, max_samples=bootstrap_frac, bootstrap=True, random_state=42, n_jobs=-1), y_pu, True),
    "lr_pu": (LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42), y_pu, True),
    "rf": (RandomForestRegressor(**RF_TUNED), y_ot, False),
    "xgb": (XGBRegressor(**XGB_TUNED), y_ot, False),
    "en": (ElasticNetCV(l1_ratio=[0.1, 0.5, 0.7, 0.9, 0.95, 1.0], cv=5, random_state=42, max_iter=10000), y_ot, False),
}

print(f"{'model':12} {'pos':>8} {'random':>8} {'neg':>8}")
for name, (model, y_target, is_classifier) in models.items():
    model.fit(X_train_s, y_target)
    test_scores = model.predict_proba(X_test_s)[:, 1] if is_classifier else model.predict(X_test_s)
    train_scores = model.predict_proba(X_train_s)[:, 1] if is_classifier else model.predict(X_train_s)

    all_scores = np.concatenate([train_scores, test_scores])
    all_gene_names = np.concatenate([train_df["target_gene"].values, test_df["target_gene"].values])
    percentile = {g: (all_scores < s).mean() * 100 for g, s in zip(all_gene_names, all_scores)}

    pos_mean = np.mean([percentile[g] for g in pos_genes if g in percentile])
    rnd_mean = np.mean([percentile[g] for g in h_random if g in percentile])
    neg_mean = np.mean([percentile[g] for g in neg_genes if g in percentile])
    print(f"{name:12} {pos_mean:>8.1f} {rnd_mean:>8.1f} {neg_mean:>8.1f}")

## Final target lists and cross-model consensus

Each of the five models is trained on all non-holdout genes and scores the full
non-holdout dataset. The 20 highest-ranked genes per model are the top ~1.5-3% of
candidates (depending on dataset size). A gene ranked in this range by at least 4 of
the 5 models (two PU classifiers and three Open Targets regression models, built on
different assumptions) is called a consensus target: agreement across that many
independent approaches is much less likely to be a single model's quirk.

In [ ]:
model_order = ["pu_bag", "lr_pu", "rf", "xgb", "en"]

def fit_and_score_all(feat_df, holdout_genes):
    """Train all 5 models on non-holdout genes, score every non-holdout gene.
    Returns per-model raw scores and percentile ranks, keyed by gene."""
    train_df = feat_df[~feat_df["target_gene"].isin(holdout_genes)].copy()
    X_train = train_df[FEATURE_COLS].fillna(0).values
    y_pu, y_ot = train_df["pu_label"].values, train_df["ot_overall"].values

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_all_s = scaler.transform(feat_df[FEATURE_COLS].fillna(0).values)

    n_pos = y_pu.sum()
    bootstrap_frac = min(1.0, (2 * n_pos) / len(train_df))

    models = {
        "pu_bag": (BaggingClassifier(
            estimator=DecisionTreeClassifier(max_depth=2, min_samples_leaf=5, class_weight="balanced"),
            n_estimators=100, max_samples=bootstrap_frac, bootstrap=True, random_state=42, n_jobs=-1), y_pu, True),
        "lr_pu": (LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42), y_pu, True),
        "rf": (RandomForestRegressor(**RF_TUNED), y_ot, False),
        "xgb": (XGBRegressor(**XGB_TUNED), y_ot, False),
        "en": (ElasticNetCV(l1_ratio=[0.1, 0.5, 0.7, 0.9, 0.95, 1.0], cv=5, random_state=42, max_iter=10000), y_ot, False),
    }

    scores, percentiles, top20 = {}, {}, {}
    for name, (model, y_target, is_classifier) in models.items():
        model.fit(X_train_s, y_target)
        all_scores = model.predict_proba(X_all_s)[:, 1] if is_classifier else model.predict(X_all_s)
        genes = feat_df["target_gene"].values

        scores[name] = dict(zip(genes, all_scores))
        percentiles[name] = {g: (all_scores < s).mean() * 100 for g, s in zip(genes, all_scores)}

        non_holdout = feat_df[~feat_df["target_gene"].isin(holdout_genes)].copy()
        non_holdout["score"] = [scores[name][g] for g in non_holdout["target_gene"]]
        top20[name] = set(non_holdout.nlargest(20, "score")["target_gene"])

    return scores, percentiles, top20

all_scores, all_percentiles, all_top20 = {}, {}, {}
for sex in ["female", "male", "combined"]:
    all_scores[sex], all_percentiles[sex], all_top20[sex] = fit_and_score_all(datasets[sex], holdout_genes)

In [ ]:
def build_consensus_table(sex):
    all_genes = set().union(*all_top20[sex].values())
    rows = []
    for gene in all_genes:
        models_with_gene = [m for m in model_order if gene in all_top20[sex][m]]
        pu_label = datasets[sex].set_index("target_gene").loc[gene, "pu_label"]
        rows.append({
            "target_gene": gene,
            "n_models": len(models_with_gene),
            "models": ", ".join(models_with_gene),
            "pu_training_label": "Positive" if pu_label else "Unlabelled",
        })
    return pd.DataFrame(rows).sort_values("n_models", ascending=False).reset_index(drop=True)

consensus_tables = {sex: build_consensus_table(sex) for sex in ["female", "male", "combined"]}

for sex, table in consensus_tables.items():
    high_consensus = table[table["n_models"] >= 4]
    print(f"\n{sex.upper()} -- {len(high_consensus)} genes at 4/5+ consensus")
    print(high_consensus.to_string(index=False))
    table.to_csv(BASE / f"consensus_targets_{sex}.csv", index=False)

## Case study percentile ranks

*ROCK2* (combined), *MAPK14* (male), and *BRD4* (female) were selected for biological case
study discussion: each is unlabelled, reaches 4/5+ consensus in at least one analysis,
and ranks consistently high across all five models individually.

In [ ]:
case_studies = [("ROCK2", "combined"), ("MAPK14", "male"), ("BRD4", "female")]
model_labels = {"pu_bag": "PU Bagging", "lr_pu": "Logistic PU", "rf": "Random Forest",
                 "xgb": "XGBoost", "en": "Elastic Net"}

print(f"{'Gene':8} {'Dataset':10} " + " ".join(f"{model_labels[m]:>13}" for m in model_order))
for gene, sex in case_studies:
    pct = all_percentiles[sex]
    values = " ".join(f"{pct[m].get(gene, float('nan')):>13.1f}" for m in model_order)
    print(f"{gene:8} {sex:10} {values}")
